In [1]:
from tqdm import tqdm
import pandas as pd
import numpy as np

import torch
from sentence_transformers import SentenceTransformer
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti


In [2]:
MODEL_NAME = "Octen/Octen-Embedding-0.6B"

model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs={
        "torch_dtype": torch.bfloat16,  # <-- This will remove the warning
        "device_map": "auto",
    },
    tokenizer_kwargs={
        "padding_side": "left",
    },
)
model.max_seq_length = 1024

The `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [3]:
BATCH_SIZE = 64
questions_df = pd.read_parquet("data/sample_questions.parquet")
question_texts = questions_df["question"].tolist()

all_question_embeddings = []

for start in tqdm(
    range(0, len(question_texts), BATCH_SIZE),
    total=(len(question_texts) + BATCH_SIZE - 1) // BATCH_SIZE,
    desc="Embedding questions",
    unit="batch",
):
    batch_texts = question_texts[start:start + BATCH_SIZE]

    batch_embeddings = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    all_question_embeddings.append(batch_embeddings)

question_embeddings = np.vstack(all_question_embeddings)
print(question_embeddings.shape)

Embedding questions: 100%|██████████| 268/268 [00:34<00:00,  7.72batch/s]

(17138, 1024)


In [5]:
embedded_questions_df = questions_df.copy()
embedded_questions_df["question_embedding"] = list(question_embeddings)

In [7]:
embedded_questions_df.to_parquet("data/encoded_questions.parquet", index=False)